# Create weighingGUI Spreadsheet daily

In [2]:
from scripts.conf_file_finding import try_find_conf_file
try_find_conf_file()

ModuleNotFoundError: No module named 'scripts'

In [ ]:
import datajoint as dj
import pandas as pd
import time
from zoneinfo import ZoneInfo
import datetime
import pathlib
import numpy as np
import time
import base64
import openpyxl

time.sleep(1)

import u19_pipeline.alert_system.water_weigh_alert.water_weigh_alert as wwa
import u19_pipeline.scheduler as scheduler
import u19_pipeline.acquisition as acquisition
import u19_pipeline.behavior as behavior
import u19_pipeline.subject as subject
import u19_pipeline.lab as lab


DJ_CUSTOM_VARIABLES_FILENAME = 'DJCustomVariables.csv'
SLACK_WEBHOOK_FILENAME = 'SlackChannels.csv'
USER_SLACK_FILENAME = 'UserSlack.csv'
RIG_STATUS_FILENAME = 'RigStatusTable.csv'
DAY_SCHEDULE_FILENAME = 'ScheduleDay.csv'
PAST_SESSION_PERFORMANCE_FILENAME = 'PastSessions.csv'



MAX_SESSIONS_HISTORY = 75

In [18]:
def cast_choice(choice_array):

    new_array = choice_array[0].copy()
    new_array[new_array>2] = 127

    new_array = np.array(new_array, dtype=np.uint8)

    return new_array


In [19]:
def encode_webhook(webhook):

    return (base64.b64encode(webhook.encode('utf-8'))).decode('utf-8')
   

In [20]:
conf = dj.config

nodb_virmen_backup_dir = pathlib.Path(pathlib.Path(conf['custom']['root_data_dir'][0]).parent.parent,'Shared','NoDBVirmenBackup')

DJ_CUSTOM_VARIABLES_FILENAME = pathlib.Path(nodb_virmen_backup_dir,DJ_CUSTOM_VARIABLES_FILENAME).as_posix()
SLACK_WEBHOOK_FILENAME = pathlib.Path(nodb_virmen_backup_dir,SLACK_WEBHOOK_FILENAME).as_posix()
USER_SLACK_FILENAME = pathlib.Path(nodb_virmen_backup_dir,USER_SLACK_FILENAME).as_posix()
RIG_STATUS_FILENAME = pathlib.Path(nodb_virmen_backup_dir,RIG_STATUS_FILENAME).as_posix()
DAY_SCHEDULE_FILENAME = pathlib.Path(nodb_virmen_backup_dir,DAY_SCHEDULE_FILENAME).as_posix()
PAST_SESSION_PERFORMANCE_FILENAME = pathlib.Path(nodb_virmen_backup_dir,PAST_SESSION_PERFORMANCE_FILENAME).as_posix()



In [21]:
pd.DataFrame(lab.DjCustomVariables.fetch(as_dict=True)).to_csv(DJ_CUSTOM_VARIABLES_FILENAME)


In [22]:
slack_webhooks = pd.DataFrame(lab.SlackWebhooks.fetch(as_dict=True))
slack_webhooks['webhook_url'] = slack_webhooks['webhook_url'].astype(str)

slack_webhooks['webhook_url'] = slack_webhooks['webhook_url'].apply(encode_webhook)
slack_webhooks.to_csv(SLACK_WEBHOOK_FILENAME, index=False)

In [23]:
user_data = pd.DataFrame(lab.User.fetch('user_id', 'slack', 'tech_responsibility', 'slack_webhook',as_dict=True))
user_data['slack_webhook'] = user_data['slack_webhook'].astype(str)
user_data['slack_webhook'] = user_data['slack_webhook'].fillna('')

user_data['slack_webhook'] = user_data['slack_webhook'].apply(encode_webhook)
user_data.to_csv(USER_SLACK_FILENAME, index=False)

In [24]:
user_data

,user_id,slack,tech_responsibility,slack_webhook
0,aarusso,Abby Russo,yes,aHR0cHM6Ly9ob29rcy5zbGFjay5jb20vc2VydmljZXMvVD...
1,ad9966,U08J0JCUEFP,N/A,
2,alvaros,UUD6UMWCS,yes,aHR0cHM6Ly9ob29rcy5zbGFjay5jb20vc2VydmljZXMvVD...
3,ao8210,U0A7SMLNM1D,yes,Tm9uZQ==
4,apv2,UDQLJ5MKM,yes,Tm9uZQ==
5,ariordan,Alex Riordan,yes,aHR0cHM6Ly9ob29rcy5zbGFjay5jb20vc2VydmljZXMvVD...
6,baptista,Scott Baptista,N/A,
7,bm9751,U06UUR8M9A6,yes,Tm9uZQ==
8,ct5868,U06GN5TBX46,no,Tm9uZQ==
9,de4003,U09KSJAVACD,yes,Tm9uZQ==


In [25]:
df_rig_status = pd.DataFrame(scheduler.RigStatus.fetch('location', 'input_output_name','current_status',as_dict=True))
df_rig_status.to_csv(RIG_STATUS_FILENAME, index=False)

In [26]:
schedule_query = dict()
schedule_query['date'] = datetime.date.today() + datetime.timedelta(days=1)

subject_query = 'subject_fullname is not null'

day_schedule = pd.DataFrame((scheduler.Schedule * scheduler.TrainingProfile * subject.Subject.proj(subject_user_id='user_id') & schedule_query & subject_query).fetch(as_dict=True))
day_schedule = day_schedule.drop(columns='user_id')
day_schedule = day_schedule.rename(columns={'subject_user_id':'user_id'})


day_schedule.to_csv(DAY_SCHEDULE_FILENAME, index=False)

In [27]:
day_schedule

,date,location,timeslot,training_profile_id,subject_fullname,recording_profile_id,input_output_profile_id,experimenters_instructions,level,sublevel,date_created,date_last_used,training_profile_name,training_profile_description,training_profile_variables,deprecated,user_id
0,2026-03-05,165A-miniVR-T-1,1,145,efonseca_ef932_act131,1,5,"Do not modify DV coordinates, unless too obvio...",0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
1,2026-03-05,165A-miniVR-T-1,2,154,efonseca_ef717_sta531,1,5,Do NOT change DV (up-down) coordinates.,0,0,2026-01-21,2026-01-21,LSTT_Stationary_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
2,2026-03-05,165A-miniVR-T-1,3,145,efonseca_ef730_act134,1,5,Do not change DV coordinates,0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
3,2026-03-05,165A-miniVR-T-2,1,145,efonseca_ef512_act132,1,5,"Do not modify DV coordinates, unless too obvio...",0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
4,2026-03-05,165A-miniVR-T-2,2,154,efonseca_ef718_sta532,1,5,Do NOT change DV (up-down) coordinates. Thanks!,0,0,2026-01-21,2026-01-21,LSTT_Stationary_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
5,2026-03-05,165A-miniVR-T-2,3,145,efonseca_ef732_act701,1,5,Do not change DV coordinates. Thanks! :D,0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
6,2026-03-05,165A-miniVR-T-3,1,145,efonseca_ef505_act133,1,5,Do not change DV (up-down) coordinates. Thanks,0,0,2025-05-06,2025-05-06,LSTT_Levels_GradualParamChange_EFv2,This is an autogenerated Profile for LSTT_Leve...,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0, 1....",0,efonseca
7,2026-03-05,165A-miniVR-T-3,2,152,efonseca_ef721_pas331,1,5,Do NOT change DV (up-down) coordinates. Thanks!,0,0,2025-10-09,2025-10-09,LSTT_Passive_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
8,2026-03-05,165A-miniVR-T-3,3,152,efonseca_ef722_pas332,1,5,Do not change DV coordinates. Thanks!,0,0,2025-10-09,2025-10-09,LSTT_Passive_TrainingProfile,,"{""rewardFactor"": [[2.0, 1.5, 1.2, 1.0, 1.0], [...",0,efonseca
9,2026-03-05,165A-miniVR-T-5,5,131,jk8386_jk112,1,2,"LASER ON\ndoric file: ""...triggered...50hz...""...",0,0,2025-04-14,2025-04-14,jesse_chronic_spatfreq_prot,Jesse's spatfreq task for optogenetics across ...,"{""rewardFactor"": [[1.0, 1.0, 1.0, 1.0, 1.0, 1....",0,jk8386


In [28]:
all_subjects_schedule = "', '".join(day_schedule['subject_fullname'])
all_subjects_schedule = "subject_fullname in ('" +all_subjects_schedule+ "')"
all_subjects_schedule

"subject_fullname in ('efonseca_ef932_act131', 'efonseca_ef717_sta531', 'efonseca_ef730_act134', 'efonseca_ef512_act132', 'efonseca_ef718_sta532', 'efonseca_ef732_act701', 'efonseca_ef505_act133', 'efonseca_ef721_pas331', 'efonseca_ef722_pas332', 'jk8386_jk112', 'jk8386_jk113', 'jounhong_DRD1cre_783', 'jounhong_DRD1cre_784', 'jounhong_DRD1cre_785', 'testuser_ef932_act131_T00', 'jeremyjc_j100', 'jeremyjc_j105', 'jeremyjc_j109', 'jeremyjc_j099', 'jeremyjc_j107', 'jeremyjc_j103', 'jyanar_ya066', 'jyanar_ya069', 'jyanar_ya070', 'jyanar_ya067', 'jyanar_ya065', 'jyanar_ya068', 'jyanar_ya071', 'jyanar_ya061', 'jyanar_ya063', 'jyanar_ya062', 'jyanar_ya064')"

In [29]:
ss = pd.DataFrame((behavior.TowersSession & all_subjects_schedule).fetch('KEY', order_by='subject_fullname, session_date desc, session_number desc', as_dict=True))
ss['session_date'] = ss['session_date'].astype(str)
ss['num_sessions'] = ss.groupby(['subject_fullname'])['subject_fullname'].rank(method='first')
ss['num_sessions'] = ss['num_sessions'].astype(int)
ss = ss.loc[ss['num_sessions'] <= MAX_SESSIONS_HISTORY, :]

#ss2 = ss.copy()

max_value_indices = ss.groupby('subject_fullname')['num_sessions'].idxmax()
ss = ss.loc[max_value_indices]
ss = ss.reset_index(drop=True)
ss['query'] = "(subject_fullname='" + ss['subject_fullname'] + "' and session_date >= '" + ss['session_date'] + "')"

block_query = ' OR '.join(ss['query'])
ss

,subject_fullname,session_date,session_number,num_sessions,query
0,efonseca_ef505_act133,2026-01-15,0,36,(subject_fullname='efonseca_ef505_act133' and ...
1,efonseca_ef512_act132,2026-01-10,0,41,(subject_fullname='efonseca_ef512_act132' and ...
2,efonseca_ef717_sta531,2026-01-29,2,23,(subject_fullname='efonseca_ef717_sta531' and ...
3,efonseca_ef718_sta532,2026-01-29,1,26,(subject_fullname='efonseca_ef718_sta532' and ...
4,efonseca_ef721_pas331,2026-01-29,0,24,(subject_fullname='efonseca_ef721_pas331' and ...
5,efonseca_ef722_pas332,2026-02-02,0,14,(subject_fullname='efonseca_ef722_pas332' and ...
6,efonseca_ef730_act134,2026-02-03,0,20,(subject_fullname='efonseca_ef730_act134' and ...
7,efonseca_ef732_act701,2026-02-03,0,19,(subject_fullname='efonseca_ef732_act701' and ...
8,efonseca_ef932_act131,2026-01-10,0,42,(subject_fullname='efonseca_ef932_act131' and ...
9,jeremyjc_j099,2025-10-20,0,75,(subject_fullname='jeremyjc_j099' and session_...


In [30]:
sstable=(acquisition.SessionStarted.proj('local_path_behavior_file', 'session_location'))
stable = (acquisition.Session).proj(stimulusBank='stimulus_bank')
tstable = (behavior.TowersSession.proj(trialType='rewarded_side',choice='chosen_side', stimulusSet='stimulus_set'))  
tbtable = (behavior.TowersBlock.proj('first_trial','n_trials', 'sublevel', mazeID='level', mainMazeID='main_level', easyBlockFlag='easy_block',\
                                     duration='block_duration', rewardMil='reward_mil', medianTrialDur='trial_duration_median', start='block_start_time'))  
table_fetch = sstable * stable* tstable * tbtable

allblocks = pd.DataFrame((table_fetch & block_query).fetch(order_by='subject_fullname, session_date desc, session_number desc, block desc',as_dict=True))
allblocks['from_DB'] = 1
allblocks['session_date'] = allblocks['session_date'].astype(str)

allblocks['sublevel'] = allblocks['sublevel'].astype('Int64')
allblocks['choice'] = allblocks['choice'].apply(cast_choice)
allblocks['trialType'] = allblocks['trialType'].apply(cast_choice)

#allblocks['session_date'] = allblocks['session_date'].astype(str)
#allblocks['num_blocks'] = allblocks.groupby(['subject_fullname'])['subject_fullname'].rank(method='first')

#all_sessions_blocks = allblocks.drop_duplicates(subset=['subject_fullname','session_date','session_number']).copy()
#all_sessions_blocks['num_session_blocks'] = all_sessions_blocks.groupby(['subject_fullname'])['subject_fullname'].rank(method='first')
#all_sessions_blocks2 = all_sessions_blocks.copy()
#max_value_indices = all_sessions_blocks.groupby('subject_fullname')['num_session_blocks'].idxmax()
#all_sessions_blocks = all_sessions_blocks.loc[max_value_indices]



 

In [31]:

allblocks.to_csv(PAST_SESSION_PERFORMANCE_FILENAME, index=False)
allblocks

,subject_fullname,session_date,session_number,block,session_location,local_path_behavior_file,stimulusBank,stimulusSet,trialType,choice,...,mazeID,sublevel,n_trials,first_trial,duration,start,rewardMil,easyBlockFlag,medianTrialDur,from_DB
0,efonseca_ef505_act133,2026-03-03,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 2, 2, 1, 1, 2, 2, 1, 2, 2, 1, 1, 1, 1, ...",...,3,1,244,1,3600.03000,2026-03-03 11:47:00,0.468,0,9.26066,1
1,efonseca_ef505_act133,2026-03-02,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1, ...",...,3,1,182,1,3600.03000,2026-03-02 11:41:00,0.376,0,9.77846,1
2,efonseca_ef505_act133,2026-02-28,0,1,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[2, 1, 2, 1, 1, 1, 2, 1, 2, 2, 1, 2, 2, 1, 1, ...","[2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, ...",...,3,1,83,1,3143.01000,2026-02-28 12:06:00,0.196,0,9.74901,1
3,efonseca_ef505_act133,2026-02-27,0,3,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 1, 2, ...",...,2,3,293,38,2960.16000,2026-02-27 10:20:00,1.168,0,6.07871,1
4,efonseca_ef505_act133,2026-02-27,0,2,165A-miniVR-T-3,C:\Data\efonseca\efonseca_ef505_act133\Session...,-- NO STIM BANK MODE --,1,"[1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 1, ...","[1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 1, 2, ...",...,2,1,1,37,5.44972,2026-02-27 10:20:00,0.004,0,5.44972,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3481,jyanar_ya071,2026-01-08,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, ...","[1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 2, ...",...,1,<NA>,10,1,203.78600,2026-01-08 10:52:00,0.080,0,13.52700,1
3482,jyanar_ya071,2026-01-07,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2, 1, 2, 1, 2, 1]","[127, 127, 127, 127, 2, 2, 127, 127, 127, 2, 1...",...,1,<NA>,13,1,3762.83000,2026-01-07 11:13:00,0.068,0,412.32000,1
3483,jyanar_ya071,2026-01-06,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2]","[127, 127, 127, 127, 127, 127, 127, 127]",...,1,<NA>,8,1,3301.60000,2026-01-06 10:53:00,0.000,0,412.77700,1
3484,jyanar_ya071,2026-01-05,0,1,165I-Rig4-T,C:\Data\jyanar\jyanar_ya071\Session_jessejorge...,C:/Experiments/ViRMEn/experiments/protocols/st...,1,"[1, 1, 1, 2, 2, 2, 1, 2, 1, 2]","[1, 1, 1, 2, 2, 2, 1, 2, 1, 127]",...,1,<NA>,10,1,1893.44000,2026-01-05 11:02:00,0.072,0,171.05800,1


In [17]:
subject_data = get_subject_data()

NameError: name 'get_subject_data' is not defined

In [15]:
pd.set_option('display.max_rows', 20)
subject_data

,subject_fullname,effective_date,subject_status,water_per_day,schedule,sr.subject_fullname,rfid,max_status.subject_fullname,last_status,s.subject_fullname,...,need_supplement,need_extra_water_now,water_status,need_water,current_need_water,training_status,training_status_label,already_weighted,need_weight,weight_status
0,efonseca_ef756_act123,2025-04-23,InExperiments,1.0,Train/Train/Train/Train/Train/Train/Train,efonseca_ef756_act123,5C8B1724,efonseca_ef756_act123,2025-04-23,efonseca_ef756_act123,...,True,0,Need Supplement,1,1.0,1,Scheduled,False,True,Need Weight
1,efonseca_ef757_act124,2025-04-23,InExperiments,1.0,Train/Train/Train/Train/Train/Train/Train,efonseca_ef757_act124,5C8B2047,efonseca_ef757_act124,2025-04-23,efonseca_ef757_act124,...,True,0,Need Supplement,1,1.0,1,Scheduled,False,True,Need Weight
2,efonseca_ef358_act125,2025-07-01,WaterRestrictionOnly,1.0,Water/Water/Water/Water/Water/Water/Water,None,None,efonseca_ef358_act125,2025-07-01,efonseca_ef358_act125,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight
3,efonseca_ef037_act126,2025-07-01,WaterRestrictionOnly,1.0,Water/Water/Water/Water/Water/Water/Water,None,None,efonseca_ef037_act126,2025-07-01,efonseca_ef037_act126,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight
4,jeremyjc_j084,2025-07-11,WaterRestrictionOnly,1.0,Nothing/Nothing/Nothing/Nothing/Nothing/Nothin...,None,None,jeremyjc_j084,2025-07-11,jeremyjc_j084,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,testuser_demo_add,2024-03-26,WaterRestrictionOnly,1.0,Train/Train/Train/Train/Train/Train/Train,None,None,testuser_demo_add,2024-03-26,testuser_demo_add,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight
102,testuser_ironman_!,2024-03-01,WaterRestrictionOnly,1.0,Train/Train/Train/Train/Train/Train/Train,None,None,testuser_ironman_!,2024-03-01,testuser_ironman_!,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight
103,testuser_monday_tria,2024-03-11,WaterRestrictionOnly,1.0,Train/Train/Train/Train/Train/Train/Train,None,None,testuser_monday_tria,2024-03-11,testuser_monday_tria,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight
104,testuser_sdfsdsaf,2025-03-19,InExperiments,1.0,Train/Train/Train/Train/Train/Train/Train,None,None,testuser_sdfsdsaf,2025-03-19,testuser_sdfsdsaf,...,True,0,Need Supplement,1,1.0,0,Not scheduled,False,True,Need Weight


In [31]:
subject_data = subject_data.loc[~subject_data['subject_fullname'].str.contains('test'),:]
subjects_not_watered = subject_data.loc[subject_data['current_need_water'] > 0, ['subject_fullname', 'current_need_water']]
subjects_not_watered = subjects_not_watered.reset_index(drop=True)
subjects_not_watered['current_need_water'] = subjects_not_watered['current_need_water'].apply(lambda x: f"{x:.1f}")
subjects_not_watered = subjects_not_watered.head()
subjects_not_weighted = subject_data.loc[subject_data['need_weight'], ['subject_fullname', 'need_weight']]
subjects_not_weighted = subjects_not_weighted.reset_index(drop=True)
subjects_not_weighted = subjects_not_weighted.head()
subjects_not_trained = subject_data.loc[subject_data['training_status']==1, ['subject_fullname', 'scheduled_rig']]
subjects_not_trained = subjects_not_trained.reset_index(drop=True)
subjects_not_trained = subjects_not_trained.head()
subjects_not_trained

,subject_fullname,scheduled_rig
0,efonseca_ef756_act123,165A-miniVR-T-3
1,efonseca_ef757_act124,165A-miniVR-T-1
2,jeremyjc_j091,165I-Rig1-T
3,jeremyjc_j092,165I-Rig1-T
4,jeremyjc_j094,165I-Rig1-T


In [32]:
lab = dj.create_virtual_module('lab', 'u19_lab')
slack_configuration_dictionary = {
    'slack_notification_channel': ['alvaro_luna']
}
webhooks_list = []
query_slack_webhooks = [{'webhook_name' : x} for x in slack_configuration_dictionary['slack_notification_channel']]
webhooks_list += (lab.SlackWebhooks & query_slack_webhooks).fetch('webhook_url').tolist()

[2025-08-21 09:38:39,228][WARNING]: Reconnecting to MySQL server.


In [35]:
def slack_alert_message_format_weight_water(subjects_not_watered, subjects_not_weighted, subjects_not_trained):

    now = datetime.datetime.now()
    datestr = now.strftime('%d-%b-%Y %H:%M:%S')

    msep = dict()
    msep['type'] = "divider"

    #Title#
    m1 = dict()
    m1['type'] = 'section'
    m1_1 = dict()
    m1_1["type"] = "mrkdwn"
    m1_1["text"] = ':rotating_light: * Subjects Status Alert *'
    m1['text'] = m1_1

    #Info#
    m2 = dict()
    m2['type'] = 'section'
    m2_1 = dict()
    m2_1["type"] = "mrkdwn"

    m2_1["text"] = '*Subjects missing water:*' + '\n'
    for i in range(subjects_not_watered.shape[0]):
        m2_1["text"] += '*' + subjects_not_watered.loc[i, 'subject_fullname'] + '* : ' + str(subjects_not_watered.loc[i, 'current_need_water']) + ' ml\n'
    #m2_1["text"] += '\n'
    m2['text'] = m2_1

    m4 = dict()
    m4['type'] = 'section'
    m4_1 = dict()
    m4_1["type"] = "mrkdwn"

    m4_1["text"] = '*Subjects missing weighing:*' + '\n'
    for i in range(subjects_not_weighted.shape[0]):
        m4_1["text"] += '*' + subjects_not_weighted.loc[i, 'subject_fullname'] + '*\n' 
    m4['text'] = m4_1

    m5 = dict()
    m5['type'] = 'section'
    m5_1 = dict()
    m5_1["type"] = "mrkdwn"

    m5_1["text"] = '*Subjects missing training:*' + '\n'
    for i in range(subjects_not_trained.shape[0]):
        m5_1["text"] += '*' + subjects_not_trained.loc[i, 'subject_fullname'] + '* : ' + subjects_not_trained.loc[i, 'scheduled_rig'] + '\n'
    #
    m5['text'] = m5_1

    message = dict()
    message['blocks'] = [m1,msep,m2,msep,m4,msep,m5,msep]
    message['text'] = 'Subject Status Alert'

    return message

In [36]:
slack_json_message = slack_alert_message_format_weight_water(subjects_not_watered, subjects_not_weighted, subjects_not_trained)


#Send alert
for this_webhook in webhooks_list:
    su.send_slack_notification(this_webhook, slack_json_message)
    time.sleep(1)


In [37]:
current_directory = pathlib.Path(__file__).resolve()

NameError: name '__file__' is not defined